In [2]:
import pickle
import torch
import torch.nn as nn
import numpy as np
import joblib


In [3]:
!pwd

/home/ylnner/Documents/new_model/scripts


In [ ]:
path = 'models/MLP_seed0.pkl'
mlp_sklearn = joblib.load(path)

In [ ]:
class TorchMLP(nn.Module):
    def __init__(self, mlp):
        super().__init__()
        layers = []
        
        for i in range(len(mlp.coefs_)):
            in_features = mlp.coefs_[i].shape[0]
            out_features = mlp.coefs_[i].shape[1]
            
            layers.append(nn.Linear(in_features, out_features))
            
            if i < len(mlp.coefs_) - 1:
                activation = mlp.activation
                print('activation', activation)
                if activation == 'relu':
                    layers.append(nn.ReLU())
                elif activation == 'tanh':
                    layers.append(nn.Tanh())
                elif activation == 'logistic':
                    layers.append(nn.Sigmoid())
                else:
                    raise ValueError(f"Activation  '{activation}' not supported")
        
        if hasattr(mlp, 'classes_'):
            if len(mlp.classes_) > 2:
                layers.append(nn.Softmax(dim=1))
            else:
                layers.append(nn.Sigmoid())
                
        self.network = nn.Sequential(*layers)

        # Transfer weights
        linear_layers = [m for m in self.network.modules() if isinstance(m, nn.Linear)]
        
        for i, linear in enumerate(linear_layers):
            # Reshape the weights : scikit-learn (in, out) -> PyTorch (out, in)            
            weight = torch.tensor(mlp.coefs_[i].T, dtype=torch.float32)
            bias = torch.tensor(mlp.intercepts_[i], dtype=torch.float32)
            
            with torch.no_grad():
                linear.weight.copy_(weight)
                linear.bias.copy_(bias)
                
    def forward(self, x):
        return self.network(x)


# Build the model
torch_model = TorchMLP(mlp_sklearn)
torch_model.eval()

# Export to pt format
input_size = mlp_sklearn.n_features_in_
ejemplo_entrada = torch.randn(1, input_size, dtype=torch.float32)

modelo_traced = torch.jit.trace(torch_model, ejemplo_entrada)
modelo_traced.save("mlp_exported.pt")

print("Modelo exportado exitosamente a mlp_exported.pt")

activation relu
activation relu
Modelo exportado exitosamente a mlp_exported.pt


In [7]:
import joblib
import torch
import numpy as np

mlp_sklearn = joblib.load(path)
modelo_pt = torch.jit.load('mlp_exported.pt')
modelo_pt.eval()

# Create synthetic data for testing
n_features = mlp_sklearn.n_features_in_
X_test = np.random.rand(5, n_features).astype(np.float32)


if hasattr(mlp_sklearn, 'classes_'):
    out_sklearn = mlp_sklearn.predict_proba(X_test)
else:
    out_sklearn = mlp_sklearn.predict(X_test)


tensor_test = torch.from_numpy(X_test)
with torch.no_grad():
    out_pytorch = modelo_pt(tensor_test).numpy()

# Compare predictions
# Reshape dimenssions if necessary
if out_sklearn.ndim == 1:
    out_sklearn = out_sklearn.reshape(-1, 1)


if out_sklearn.shape[1] == 2 and out_pytorch.shape[1] == 1:
    out_sklearn = out_sklearn[:, 1:2] 

diferencia_maxima = np.max(np.abs(out_sklearn - out_pytorch))

print("=== Resultados de la Validación ===")
print(f"Salida scikit-learn (Ajustada): {out_sklearn[0]}")
print(f"Salida TorchScript            : {out_pytorch[0]}")
print("-" * 33)
print(f"Diferencia máxima observada: {diferencia_maxima:.8e}")

if diferencia_maxima < 1e-4:
    print("Predictions are equal within the tolerance of 1e-4.")
else:
    print("ERROR-Difference between predictions ")

=== Resultados de la Validación ===
Salida scikit-learn (Ajustada): [0.85260859]
Salida TorchScript            : [0.8526085]
---------------------------------
Diferencia máxima observada: 8.54105527e-08
Predictions are equal within the tolerance of 1e-4.


# Get the scaler

In [6]:
!pwd

/home/ylnner/Documents/new_model/scripts


In [7]:
import json
import joblib  # Using joblib to match the actual file formatting

# Load your scaler using joblib instead of pickle
# Pass the filename directly to joblib.load
model_path = 'models/scaler_seed0.pkl'
scaler = joblib.load(model_path)

# Extract the mathematical attributes
# Note: If this is a MinMaxScaler instead of StandardScaler, 
# you'll want to use: "min": scaler.data_min_.tolist(), "max": scaler.data_max_.tolist()
scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}

# Save it to a clean JSON file for OMNeT++
with open("scaler_params.json", "w") as f:
    json.dump(scaler_data, f)

print("Extraction successful!")
scaler_data

Extraction successful!


{'mean': [48.59192333333333,
  11.996563166666666,
  9.85826152279762,
  0.0204108,
  11.116666666666667,
  -582.7580839379167,
  715.0,
  214.82142857142858],
 'scale': [8.249025798831575,
  13.106201991101578,
  17.685909582817963,
  0.0039332991190602325,
  0.7765665171481163,
  14675.439418645408,
  88.94219631712659,
  75.23750659052428]}